# Online Metrics untuk LLM — Serving + MySQL + FastAPI + Streamlit

Revisi dari versi sebelumnya. Perubahan yang **wajib** diperhatikan sebelum
menjalankan ulang:

1. **Kredensial dipindah ke Colab Secrets** (fallback input manual). Versi
   sebelumnya menaruh password database dan authtoken ngrok langsung di
   kode — kalau notebook itu sudah pernah dibagikan/diunggah, anggap
   keduanya bocor dan **ganti password database + reset authtoken ngrok**
   sebelum lanjut.
2. `/regenerate` diperbaiki — versi sebelumnya punya `return history` di
   awal fungsi yang membuat sisa kode di bawahnya tidak pernah jalan, dan
   `chat_history` tidak pernah menyimpan `input_data` sehingga prompt
   regenerasi dibangun dari data yang salah.
3. Cache fuzzy diperbaiki — versi sebelumnya memakai `fuzz.partial_ratio`
   (cocok berdasarkan substring) dan mengembalikan kecocokan pertama di
   atas threshold, bukan yang terbaik. Ini pernah menyebabkan pertanyaan
   "presiden ketiga" dijawab dari cache "presiden pertama".
4. Reaksi dislike/regenerate sekarang menghapus entri cache terkait, supaya
   jawaban yang sudah ditandai kurang bagus tidak disajikan lagi ke
   pertanyaan yang mirip.
5. Ditambah rate (bukan cuma jumlah mentah) dan latency per respons, sesuai
   istilah di materi (Like/Dislike Rate, Latency).
6. Dashboard Streamlit sekarang menampilkan tren harian sungguhan, bukan
   satu angka yang diulang 30 kali.
7. `n_ctx` dinaikkan dari 512 ke 2048.
8. Skema tabel dirapikan (komentar sisa sesi lain seperti `VECTOR_SIZE`,
   `face_embeddings`, dan import `psycopg2` dihapus; foreign key diperbaiki
   supaya benar-benar dibuat oleh MySQL).

9. **Model diunduh langsung dari Hugging Face**, bukan diasumsikan sudah ada
   di Drive. Path Drive di notebook sebelumnya (`unsloth.Q4_K_M.gguf`) cuma
   ada di Drive pembuat materinya, bukan otomatis ada di Drive siapa pun
   yang menjalankan notebook ini -- itu sebabnya muncul error
   `Model path does not exist`. Model penggantinya diunduh sekali lalu
   disimpan ke Drive supaya sesi berikutnya tidak unduh ulang.
10. **Prompt tidak lagi disusun manual pakai template Alpaca** -- sekarang
    memakai `chat_format="llama-3"` bawaan `llama-cpp-python`, supaya format
    prompt otomatis cocok dengan cara model instruct di-fine-tune. Token
    sisa seperti `[/INST]` di versi sebelumnya kemungkinan besar muncul
    karena template prompt manual tidak cocok dengan model.

> **Provenance model:** model asli yang disebut di notebook sebelumnya
> (`rubythalib33/llama3_1_8b_finetuned_bahasa_indonesia`) tidak ditemukan
> sebagai repo publik saat notebook ini disusun ulang, jadi diganti dengan
> `gmonsoon/llama3-8b-cpt-sahabatai-v1-instruct-GGUF` -- fine-tune Llama 3
> 8B Instruct untuk Bahasa Indonesia oleh GoTo Group & Indosat Ooredoo
> Hutchison (lisensi Llama 3, publik). Kalau kamu punya/menemukan model
> asli dari materi bootcamp, cukup ganti `HF_REPO_ID`/`HF_FILENAME` di
> Bagian 2.

## 1. Setup Database (MySQL)

Kredensial diambil dari Colab Secrets (ikon kunci di sidebar kiri), dengan
fallback `getpass` kalau secret belum di-set. Jangan hardcode nilai apa pun
di sel ini.

In [ ]:
!pip install -q mysql-connector-python

In [ ]:
import mysql.connector
from getpass import getpass

def get_secret(name, prompt):
    # .strip() jaga-jaga kalau nilainya ke-copy-paste dengan baris baru/spasi
    # nyangkut di ujung (pernah kejadian: DB_NAME jadi "freedb_xxx\r\n").
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    return getpass(prompt).strip()

DB_NAME = get_secret("DB_NAME", "Nama database MySQL: ")
DB_USER = get_secret("DB_USER", "User MySQL: ")
DB_PASSWORD = get_secret("DB_PASSWORD", "Password MySQL: ")
DB_HOST = get_secret("DB_HOST", "Host MySQL: ")
DB_PORT = int(get_secret("DB_PORT", "Port MySQL (contoh 3306): ") or 3306)

In [ ]:
# ===== Langkah 1: Koneksi ke server & buat database jika belum ada =====
try:
    connect_server = mysql.connector.connect(
        host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD
    )
    connect_cursor = connect_server.cursor()
    connect_cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
    print(f"Database `{DB_NAME}` sudah tersedia atau berhasil dibuat.")
    connect_cursor.close()
    connect_server.close()
except Exception as e:
    print("Gagal membuat atau mengecek database:", e)

# ===== Langkah 2: Koneksi ke database & buat tabel =====
# Catatan skema (perbaikan dari versi sebelumnya):
# - chat_history sekarang menyimpan input_data juga, supaya /regenerate bisa
#   membangun ulang prompt (instruction + input) dengan benar, bukan
#   menyalahgunakan kolom answer sebagai input seperti versi sebelumnya.
# - analytics.chat_history_id bertipe sama dengan chat_history.id (SERIAL =
#   BIGINT UNSIGNED di MySQL), dan foreign key-nya dideklarasikan eksplisit
#   lewat FOREIGN KEY (...) REFERENCES (...) -- sintaks inline
#   "col REFERENCES tbl(col)" di level kolom diterima parser MySQL tapi
#   TIDAK benar-benar membuat constraint-nya.
# - latency_ms disimpan per respons untuk metrik Latency di materi.
try:
    conn = mysql.connector.connect(
        host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME
    )
    cur = conn.cursor()

    cur.execute('''
        CREATE TABLE IF NOT EXISTS cached_chat (
            id SERIAL PRIMARY KEY,
            instruction TEXT NOT NULL,
            input_data TEXT NOT NULL,
            response TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    ''')

    cur.execute('''
        CREATE TABLE IF NOT EXISTS chat_history (
            id SERIAL PRIMARY KEY,
            chat TEXT NOT NULL,
            input_data TEXT NOT NULL,
            answer TEXT NOT NULL,
            latency_ms INT DEFAULT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
    ''')

    cur.execute('''
        CREATE TABLE IF NOT EXISTS analytics (
            id SERIAL PRIMARY KEY,
            type VARCHAR(50) NOT NULL,
            chat_history_id BIGINT UNSIGNED NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (chat_history_id) REFERENCES chat_history(id) ON DELETE CASCADE
        );
    ''')

    conn.commit()
    print("Tabel cached_chat, chat_history, analytics berhasil dibuat atau sudah tersedia.")
except Exception as e:
    print("Gagal membuat tabel:", e)
finally:
    if 'cur' in locals():
        cur.close()
    if 'conn' in locals():
        conn.close()

In [ ]:
def display_database_contents():
    # Menampilkan seluruh tabel dan isi datanya dari database MySQL.
    conn = None
    try:
        conn = mysql.connector.connect(
            host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME
        )
        cursor = conn.cursor()
        cursor.execute("SHOW TABLES;")
        tables = [row[0] for row in cursor.fetchall()]

        if not tables:
            print("Tidak ada tabel dalam database.")
            return

        print(f"Daftar tabel di database `{DB_NAME}`: {', '.join(tables)}")
        for table_name in tables:
            print(f"\n--- Isi tabel: {table_name} ---")
            cursor.execute(f"SELECT * FROM {table_name}")
            rows = cursor.fetchall()
            if not rows:
                print("  (tidak ada data)")
            else:
                columns = [desc[0] for desc in cursor.description]
                print(" | ".join(columns))
                for row in rows:
                    print(" | ".join(map(str, row)))
        cursor.close()
    except Exception as e:
        print(f"Error saat menampilkan isi database: {e}")
    finally:
        if conn:
            conn.close()

display_database_contents()

## 2. Setup LLM (llama.cpp)

`n_ctx` dinaikkan ke 2048. Kalau model GGUF-mu adalah versi instruct/chat
(bukan base model), pertimbangkan memakai `chat_format` bawaan
`llama-cpp-python` (misal `chat_format="llama-3"`) alih-alih menyusun
prompt Alpaca manual — token sisa seperti `[/INST]` yang kadang muncul di
output biasanya tanda template prompt tidak cocok dengan cara model
di-fine-tune.

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio llama-cpp-python

In [ ]:
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel
from llama_cpp import Llama
from typing import Optional
from datetime import datetime
import time
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from threading import Thread

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# Model publik untuk Bahasa Indonesia -- lihat catatan provenance di atas.
# Ganti HF_REPO_ID/HF_FILENAME kalau kamu punya model lain (mis. model asli
# dari materi bootcamp, kalau kamu menemukannya).
HF_REPO_ID = "gmonsoon/llama3-8b-cpt-sahabatai-v1-instruct-GGUF"
HF_FILENAME = "llama3-8b-cpt-sahabatai-v1-instruct.Q4_K_M.gguf"  # ~4.9 GB

DRIVE_MODEL_DIR = "/content/drive/MyDrive/llama_model"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
drive_model_path = os.path.join(DRIVE_MODEL_DIR, HF_FILENAME)

# llama-cpp-python membaca model dengan pola akses (mmap/random read) yang
# SANGAT lambat kalau langsung dari Google Drive (mount-nya lewat FUSE) --
# bisa berujung "loading" puluhan menit. Drive tetap dipakai sebagai cache
# supaya tidak unduh ulang tiap sesi, tapi model SELALU disalin dulu ke
# disk lokal Colab (/content, bukan /content/drive) sebelum di-load.
LOCAL_MODEL_DIR = "/content/local_model"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
local_model_path = os.path.join(LOCAL_MODEL_DIR, HF_FILENAME)

if os.path.exists(local_model_path):
    model_path = local_model_path
    print(f"Model sudah ada di disk lokal Colab: {model_path}")
elif os.path.exists(drive_model_path):
    print("Model ada di Drive, menyalin ke disk lokal Colab dulu (sekali per sesi runtime)...")
    shutil.copy(drive_model_path, local_model_path)
    model_path = local_model_path
    print(f"Selesai disalin ke disk lokal: {model_path}")
else:
    from huggingface_hub import hf_hub_download
    print("Model belum ada di Drive, mengunduh dari Hugging Face (sekali saja, ~4.9GB)...")
    downloaded_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_FILENAME)
    shutil.copy(downloaded_path, drive_model_path)   # simpan ke Drive untuk sesi berikutnya
    shutil.copy(downloaded_path, local_model_path)   # dan langsung siap dipakai di sesi ini
    model_path = local_model_path
    print(f"Model diunduh, disimpan ke Drive, dan disalin ke disk lokal: {model_path}")

In [ ]:
nest_asyncio.apply()
app = FastAPI()

llm = Llama(
    model_path=model_path,
    n_ctx=2048,       # dinaikkan dari 512 -- terlalu kecil untuk percakapan multi-giliran
    n_batch=512,
    chat_format="llama-3",  # biar template chat otomatis cocok, bukan disusun manual
)

In [ ]:
class ChatRequest(BaseModel):
    instruction: str
    input_data: str = ""

class ChatResponse(BaseModel):
    response: str
    chat_history_id: int
    latency_ms: int

class ReactionRequest(BaseModel):
    chat_history_id: int
    reaction: str

class RegenerateRequest(BaseModel):
    chat_history_id: int

def build_user_content(instruction, input_data):
    # chat_format="llama-3" di Llama() sudah menangani template chat --
    # di sini cukup gabungkan instruction + input jadi satu pesan user,
    # tanpa template Alpaca manual.
    if input_data:
        return f"{instruction}\n\n{input_data}"
    return instruction

## 3. Fungsi Utilitas Database

Perbaikan di sini:

- `get_cached_response` sekarang mencari skor tertinggi di antara semua
  kandidat (bukan mengembalikan kandidat pertama yang lolos threshold), dan
  memakai `fuzz.ratio` (bandingkan keseluruhan string) alih-alih
  `fuzz.partial_ratio` (cocok kalau salah satu jadi substring dari yang
  lain — inilah yang tadinya membuat "apa itu algoritma" cocok 94% dengan
  "Apa itu algoritma pemrograman?" padahal pertanyaannya beda).
- Threshold default dinaikkan dari 80 ke 92. Sudah dicoba angka 85 dulu,
  tapi ternyata belum cukup aman: "Siapa presiden ketiga di indonesia" vs
  "siapa presiden pertama di indonesia?" (dua pertanyaan yang jelas beda)
  masih dapat skor 85.7 dengan `fuzz.ratio` -- di atas threshold lama.
- **Keterbatasan yang masih ada dan perlu disadari:** fuzzy matching
  berbasis karakter tidak bisa diandalkan membedakan dua pertanyaan yang
  cuma beda satu kata kunci pendek/angka kalau sisa kalimatnya sama persis.
  Contoh: "siapa presiden ke-2 di konoha" vs "siapa presiden ke-4 di
  konoha" tetap dapat skor ~96.5 -- nyaris sama dengan skor dua kalimat
  yang benar-benar identik (~97-98). Tidak ada nilai threshold yang bisa
  memisahkan keduanya dengan bersih. Kalau kasus seperti ini penting untuk
  aplikasimu, pertimbangkan pengecekan tambahan (misal ekstraksi
  angka/entitas sebelum mempercayai cache hit, atau threshold cosine
  similarity yang sangat tinggi dari sentence embedding) -- di luar
  cakupan revisi kali ini.
- `save_chat_history` sekarang menyimpan `input_data` dan `latency_ms`.
- Ditambah `invalidate_cache_for` yang dipanggil saat reaksi dislike atau
  regenerate, supaya cache tidak terus menyajikan jawaban yang sudah
  ditandai kurang bagus.

In [ ]:
!pip install -q rapidfuzz

In [ ]:
import mysql.connector
from typing import Optional
from rapidfuzz import fuzz


def get_connection():
    return mysql.connector.connect(
        host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME
    )


# ============================================
# Fungsi Cache
# ============================================
def get_cached_response(instruction, input_data, threshold=92):
    # Ambil kandidat dengan skor TERTINGGI di atas threshold, bukan
    # kandidat pertama yang lolos seperti versi sebelumnya.
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT instruction, input_data, response FROM cached_chat")
        cached_entries = cursor.fetchall()

        best_match = None
        best_score = -1

        for cached_instruction, cached_input_data, cached_response in cached_entries:
            if input_data != cached_input_data:
                continue
            score = fuzz.ratio(instruction, cached_instruction)
            if score > best_score:
                best_score = score
                best_match = cached_response

        if best_match is not None and best_score >= threshold:
            print(f"Cache hit (skor tertinggi {best_score:.1f}, threshold {threshold})")
            return best_match

        return None
    except mysql.connector.Error as e:
        print(f"DB error (get_cached_response): {e}")
        return None
    finally:
        if conn:
            conn.close()


def cache_response(instruction, input_data, response):
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO cached_chat (instruction, input_data, response) VALUES (%s, %s, %s)",
            (instruction, input_data, response),
        )
        conn.commit()
        print("Cache disimpan.")
    except mysql.connector.Error as e:
        print(f"DB error (cache_response): {e}")
    finally:
        if conn:
            conn.close()


def invalidate_cache_for(instruction, input_data):
    # Dipanggil saat dislike/regenerate -- hapus entri cache yang cocok
    # persis, supaya jawaban yang sudah ditandai kurang bagus tidak
    # disajikan lagi untuk pertanyaan yang sama.
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            "DELETE FROM cached_chat WHERE instruction = %s AND input_data = %s",
            (instruction, input_data),
        )
        conn.commit()
        if cursor.rowcount:
            print(f"Cache untuk instruksi ini dihapus ({cursor.rowcount} baris).")
    except mysql.connector.Error as e:
        print(f"DB error (invalidate_cache_for): {e}")
    finally:
        if conn:
            conn.close()


# ============================================
# Fungsi Chat History
# ============================================
def save_chat_history(chat, input_data, answer, latency_ms):
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO chat_history (chat, input_data, answer, latency_ms) VALUES (%s, %s, %s, %s)",
            (chat, input_data, answer, latency_ms),
        )
        conn.commit()
        chat_history_id = cursor.lastrowid
        print(f"Chat history disimpan (ID: {chat_history_id})")
        return chat_history_id
    except mysql.connector.Error as e:
        print(f"DB error (save_chat_history): {e}")
        return None
    finally:
        if conn:
            conn.close()


def get_chat_history_by_id(chat_history_id):
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            "SELECT chat, input_data, answer FROM chat_history WHERE id = %s",
            (chat_history_id,),
        )
        return cursor.fetchone()
    except mysql.connector.Error as e:
        print(f"DB error (get_chat_history_by_id): {e}")
        return None
    finally:
        if conn:
            conn.close()


# ============================================
# Fungsi Analytics (Reaksi + Rate)
# ============================================
def add_reaction(chat_history_id, reaction):
    # Untuk dislike/regenerate, cache terkait ikut dihapus supaya tidak
    # terus disajikan.
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO analytics (type, chat_history_id) VALUES (%s, %s)",
            (reaction, chat_history_id),
        )
        conn.commit()
        print("Reaksi berhasil ditambahkan.")
    except mysql.connector.Error as e:
        print(f"DB error (add_reaction): {e}")
    finally:
        if conn:
            conn.close()

    if reaction in ("dislike", "regenerate"):
        history = get_chat_history_by_id(chat_history_id)
        if history:
            chat, input_data, _answer = history
            invalidate_cache_for(chat, input_data)


def get_total_reactions(reaction_type=None, start_datetime=None, end_datetime=None):
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor(dictionary=True)
        query = "SELECT COUNT(*) AS total FROM analytics WHERE 1=1"
        params = []
        if reaction_type:
            query += " AND type = %s"
            params.append(reaction_type)
        if start_datetime:
            query += " AND created_at >= %s"
            params.append(start_datetime)
        if end_datetime:
            query += " AND created_at <= %s"
            params.append(end_datetime)
        cursor.execute(query, params)
        result = cursor.fetchone()
        return int(result["total"]) if result else 0
    except mysql.connector.Error as e:
        print(f"DB error (get_total_reactions): {e}")
        return 0
    finally:
        if conn:
            conn.close()


def get_reaction_rate(reaction_type, start_datetime=None, end_datetime=None):
    # Rate = jumlah reaksi tipe tertentu / jumlah total chat pada rentang
    # waktu yang sama. Ini yang membedakan "Like/Dislike Rate" di materi
    # dari sekadar hitungan mentah.
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor(dictionary=True)

        chat_query = "SELECT COUNT(*) AS total FROM chat_history WHERE 1=1"
        params = []
        if start_datetime:
            chat_query += " AND created_at >= %s"
            params.append(start_datetime)
        if end_datetime:
            chat_query += " AND created_at <= %s"
            params.append(end_datetime)
        cursor.execute(chat_query, params)
        total_chat = cursor.fetchone()["total"]

        if total_chat == 0:
            return 0.0

        total_reaction = get_total_reactions(reaction_type, start_datetime, end_datetime)
        return total_reaction / total_chat
    except mysql.connector.Error as e:
        print(f"DB error (get_reaction_rate): {e}")
        return 0.0
    finally:
        if conn:
            conn.close()


def get_reactions_by_day(reaction_type=None):
    # Jumlah reaksi per hari -- dipakai dashboard Streamlit supaya
    # grafiknya menunjukkan tren sungguhan, bukan satu angka diulang.
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor(dictionary=True)
        query = "SELECT DATE(created_at) AS day, COUNT(*) AS total FROM analytics WHERE 1=1"
        params = []
        if reaction_type:
            query += " AND type = %s"
            params.append(reaction_type)
        query += " GROUP BY DATE(created_at) ORDER BY day"
        cursor.execute(query, params)
        return cursor.fetchall()
    except mysql.connector.Error as e:
        print(f"DB error (get_reactions_by_day): {e}")
        return []
    finally:
        if conn:
            conn.close()

## 4. Endpoint FastAPI

Perbaikan utama: `/regenerate` tidak lagi punya `return` di baris pertama
(bug ini membuat sisa fungsinya mati total), dan sekarang membangun ulang
prompt dari `(instruction, input_data)` yang benar, bukan menyalahgunakan
`(chat, answer)` seperti versi sebelumnya. Latency diukur di sekitar
pemanggilan model.

In [ ]:
@app.post("/chat", response_model=ChatResponse)
async def chat_completion(request: ChatRequest):
    start = time.perf_counter()
    cached = get_cached_response(request.instruction, request.input_data)
    if cached:
        latency_ms = int((time.perf_counter() - start) * 1000)
        chat_history_id = save_chat_history(request.instruction, request.input_data, cached, latency_ms)
        return ChatResponse(response=cached, chat_history_id=chat_history_id, latency_ms=latency_ms)

    user_content = build_user_content(request.instruction, request.input_data)
    result = llm.create_chat_completion(messages=[{"role": "user", "content": user_content}])
    response_text = result["choices"][0]["message"]["content"]
    latency_ms = int((time.perf_counter() - start) * 1000)

    cache_response(request.instruction, request.input_data, response_text)
    chat_history_id = save_chat_history(request.instruction, request.input_data, response_text, latency_ms)
    return ChatResponse(response=response_text, chat_history_id=chat_history_id, latency_ms=latency_ms)


@app.post("/regenerate", response_model=ChatResponse)
async def regenerate_chat(request: RegenerateRequest):
    history = get_chat_history_by_id(request.chat_history_id)
    if not history:
        raise HTTPException(status_code=404, detail="Chat history not found")

    instruction, input_data, _old_answer = history
    start = time.perf_counter()
    user_content = build_user_content(instruction, input_data)
    result = llm.create_chat_completion(messages=[{"role": "user", "content": user_content}])
    response_text = result["choices"][0]["message"]["content"]
    latency_ms = int((time.perf_counter() - start) * 1000)

    new_id = save_chat_history(instruction, input_data, response_text, latency_ms)
    add_reaction(request.chat_history_id, "regenerate")  # ini juga menghapus cache lama
    return ChatResponse(response=response_text, chat_history_id=new_id, latency_ms=latency_ms)


@app.post("/react")
async def react_to_chat(request: ReactionRequest):
    if request.reaction not in ["like", "dislike"]:
        raise HTTPException(status_code=400, detail="Invalid reaction.")
    add_reaction(request.chat_history_id, request.reaction)
    return {"message": "Reaction saved successfully."}


@app.get("/total-reactions")
async def get_total_reaction_count(
    reaction_type: Optional[str] = Query(None),
    start_datetime: Optional[str] = Query(None),
    end_datetime: Optional[str] = Query(None),
):
    total = get_total_reactions(reaction_type, start_datetime, end_datetime)
    return {"total_reactions": total}


@app.get("/reaction-rate")
async def get_reaction_rate_endpoint(
    reaction_type: str = Query(...),
    start_datetime: Optional[str] = Query(None),
    end_datetime: Optional[str] = Query(None),
):
    rate = get_reaction_rate(reaction_type, start_datetime, end_datetime)
    return {"reaction_type": reaction_type, "rate": rate}


@app.get("/reactions-by-day")
async def get_reactions_by_day_endpoint(reaction_type: Optional[str] = Query(None)):
    return {"data": get_reactions_by_day(reaction_type)}

## 5. Uji Lokal (tanpa lewat HTTP)

Fungsi bantu untuk uji manual di dalam notebook sebelum expose lewat
ngrok.

In [ ]:
def chat_completion_local(instruction, input_data="", threshold=92):
    start = time.perf_counter()
    cached = get_cached_response(instruction, input_data, threshold)
    if cached:
        latency_ms = int((time.perf_counter() - start) * 1000)
        chat_history_id = save_chat_history(instruction, input_data, cached, latency_ms)
        if chat_history_id is None:
            raise RuntimeError(
                "Gagal menyimpan ke chat_history (lihat pesan 'DB error' di atas). "
                "Cek apakah tabelnya sudah dibuat -- jalankan ulang cell "
                "'Setup Database (MySQL)' di Bagian 1."
            )
        print(f"[cache] {instruction} -> {cached}")
        return ChatResponse(response=cached, chat_history_id=chat_history_id, latency_ms=latency_ms)

    user_content = build_user_content(instruction, input_data)
    result = llm.create_chat_completion(messages=[{"role": "user", "content": user_content}])
    response_text = result["choices"][0]["message"]["content"]
    latency_ms = int((time.perf_counter() - start) * 1000)

    cache_response(instruction, input_data, response_text)
    chat_history_id = save_chat_history(instruction, input_data, response_text, latency_ms)
    if chat_history_id is None:
        raise RuntimeError(
            "Gagal menyimpan ke chat_history (lihat pesan 'DB error' di atas). "
            "Cek apakah tabelnya sudah dibuat -- jalankan ulang cell "
            "'Setup Database (MySQL)' di Bagian 1."
        )
    return ChatResponse(response=response_text, chat_history_id=chat_history_id, latency_ms=latency_ms)

In [ ]:
def regenerate_local(chat_history_id):
    # Versi lokal dari endpoint /regenerate -- membangun ulang prompt dari
    # (instruction, input_data) yang tersimpan, lalu mencatat reaksi
    # "regenerate" (yang juga menghapus cache lama untuk pertanyaan itu).
    history = get_chat_history_by_id(chat_history_id)
    if not history:
        raise RuntimeError(f"chat_history_id {chat_history_id} tidak ditemukan.")
    instruction, input_data, _old_answer = history

    start = time.perf_counter()
    user_content = build_user_content(instruction, input_data)
    result = llm.create_chat_completion(messages=[{"role": "user", "content": user_content}])
    response_text = result["choices"][0]["message"]["content"]
    latency_ms = int((time.perf_counter() - start) * 1000)

    new_id = save_chat_history(instruction, input_data, response_text, latency_ms)
    if new_id is None:
        raise RuntimeError(
            "Gagal menyimpan hasil regenerasi ke chat_history. "
            "Cek apakah tabelnya sudah dibuat -- jalankan ulang cell "
            "'Setup Database (MySQL)' di Bagian 1."
        )
    add_reaction(chat_history_id, "regenerate")  # ini juga menghapus cache lama
    return ChatResponse(response=response_text, chat_history_id=new_id, latency_ms=latency_ms)

In [ ]:
# Uji 1: pertanyaan baru (belum ada di cache) -> lewat ke model, lalu ke-cache
resp1 = chat_completion_local("Siapa presiden pertama Indonesia?")
resp1

In [ ]:
# Uji 2: pertanyaan yang MIRIP tapi BUKAN pertanyaan yang sama -- seharusnya
# TIDAK cache-hit ke jawaban Uji 1. Ini persis kasus yang dulu salah
# dijawab dari cache "presiden pertama" di versi sebelumnya (partial_ratio +
# ambil kandidat pertama di atas threshold 80 lolos di skor 88.2). Dengan
# fuzz.ratio + threshold 92, skornya jadi 85.7 -- di bawah threshold, jadi
# sekarang lewat ke model, bukan cache.
resp2 = chat_completion_local("Siapa presiden ketiga Indonesia?")
resp2

In [ ]:
# Uji 3: ulangi PERSIS pertanyaan Uji 1 -- sekarang SEHARUSNYA cache-hit,
# karena sudah pernah ditanya dan di-cache di Uji 1.
resp3 = chat_completion_local("Siapa presiden pertama Indonesia?")
resp3

In [ ]:
# Uji reaksi: kirim dislike untuk jawaban di Uji 1, lalu cek dua hal --
# (1) baris baru muncul di tabel analytics, dan (2) entri cache untuk
# pertanyaan itu ikut terhapus (invalidate_cache_for terpanggil).
add_reaction(resp1.chat_history_id, "dislike")
display_database_contents()

In [ ]:
# Uji reaksi: kirim like untuk jawaban di Uji 2 ("presiden ketiga") --
# instruksi yang beda dari yang dipakai uji dislike, supaya tidak saling
# menghapus cache satu sama lain.
add_reaction(resp2.chat_history_id, "like")
display_database_contents()

In [ ]:
# Uji reaksi: regenerasi jawaban di Uji 3 -- ini juga otomatis mencatat
# reaksi "regenerate" untuk chat_history_id lama, dan menyimpan jawaban
# baru sebagai baris chat_history terpisah.
resp_regen = regenerate_local(resp3.chat_history_id)
print(resp_regen)
display_database_contents()

## 6. Expose lewat ngrok

Authtoken diambil dari Colab Secrets, bukan hardcode. Tunnel ini berguna
untuk menguji endpoint `/chat` dkk dari luar Colab (mis. lewat Postman atau
curl) -- tapi kalau kamu berencana lanjut ke Bagian 7 (Dashboard
Streamlit), tunnel ini akan otomatis diputus di sana (akun ngrok gratis
kadang cuma mendukung 1 tunnel publik aktif). Server FastAPI-nya sendiri
tetap jalan terus, cuma akses publiknya yang ditutup.

In [ ]:
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", "ngrok authtoken: ")
ngrok.set_auth_token(NGROK_AUTHTOKEN)

In [ ]:
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run)
thread.start()

## 7. Dashboard Streamlit

Perbedaan dari versi sebelumnya: grafiknya sekarang mengambil data per-hari
lewat `/reactions-by-day` (jadi benar-benar menunjukkan tren), dan
menampilkan rate lewat `/reaction-rate`, bukan cuma jumlah mentah.

**Kenapa `PUBLIC_API_URL` di `app.py` di-hardcode ke `http://localhost:8000`,
bukan URL ngrok:** FastAPI dan Streamlit jalan di VM Colab yang sama, jadi
Streamlit bisa panggil FastAPI langsung lewat localhost, tanpa lewat
internet publik. Ini juga menghindari masalah nyata yang pernah kejadian --
akun ngrok gratis kadang hanya mengizinkan 1 tunnel publik aktif, jadi kalau
FastAPI dan Streamlit sama-sama minta tunnel sendiri, salah satunya bisa
kebagian hostname yang sama dan permintaan API nyasar balik ke Streamlit.
Tunnel ngrok di sel berikutnya (Bagian ini) cuma untuk buka dashboard-nya di
browser kamu -- bukan untuk komunikasi Streamlit-ke-FastAPI.

In [ ]:
!pip install -q streamlit pyngrok pandas requests

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import requests
from datetime import datetime, timedelta

# FastAPI dan Streamlit jalan di VM Colab yang sama, jadi panggil langsung
# lewat localhost -- tidak lewat tunnel ngrok publik. Ini sengaja, supaya
# tidak tabrakan dengan tunnel ngrok milik Streamlit sendiri (lihat catatan
# di Bagian 7).
PUBLIC_API_URL = "http://localhost:8000"

st.title("Online Metrics Monitoring Dashboard")

reaction_type = st.selectbox(
    "Select Reaction Type:",
    options=["like", "dislike", "regenerate"],
    index=1,
)

start_date = st.date_input("Select Start Date:", value=datetime.now() - timedelta(days=30))
end_date = st.date_input("Select End Date:", value=datetime.now()) + timedelta(days=1)
start_date_str = start_date.strftime("%Y-%m-%d %H:%M:%S")
end_date_str = end_date.strftime("%Y-%m-%d %H:%M:%S")


def fetch_json(path, params=None):
    try:
        response = requests.get(f"{PUBLIC_API_URL}{path}", params=params or {})
        if response.status_code != 200:
            st.error(f"Gagal memanggil {path}: status {response.status_code} -- {response.text[:200]}")
            return None
        try:
            return response.json()
        except ValueError:
            st.error(f"Respons {path} bukan JSON valid (status 200). Isi mentah: {response.text[:200]!r}")
            return None
    except Exception as e:
        st.error(f"Error memanggil {path}: {e}")
    return None


total_data = fetch_json(
    "/total-reactions",
    {"reaction_type": reaction_type, "start_datetime": start_date_str, "end_datetime": end_date_str},
)
rate_data = fetch_json(
    "/reaction-rate",
    {"reaction_type": reaction_type, "start_datetime": start_date_str, "end_datetime": end_date_str},
)

col1, col2 = st.columns(2)
if total_data is not None:
    col1.metric(f"Total {reaction_type}", total_data["total_reactions"])
if rate_data is not None:
    col2.metric(f"{reaction_type.capitalize()} rate", f"{rate_data['rate']:.1%}")

daily = fetch_json("/reactions-by-day", {"reaction_type": reaction_type})
if daily and daily.get("data"):
    df = pd.DataFrame(daily["data"])
    df["day"] = pd.to_datetime(df["day"])
    st.bar_chart(df.set_index("day")["total"])
else:
    st.info("Belum ada data reaksi untuk rentang tanggal ini.")

In [ ]:
from pyngrok import ngrok
import threading
import time
import os

# Akun ngrok gratis kadang cuma bisa 1 tunnel publik yang benar-benar aktif
# (dua-duanya kebagian hostname yang sama, salah satu jadi tidak nyampai).
# Karena app.py sekarang panggil FastAPI lewat localhost (tidak butuh
# tunnel publiknya lagi), putuskan tunnel FastAPI dari Bagian 6 di sini --
# server FastAPI-nya TETAP jalan (cuma tunnel publiknya yang ditutup),
# jadi localhost:8000 masih bisa diakses Streamlit seperti biasa.
try:
    ngrok.disconnect(public_url.public_url)
    print(f"Tunnel FastAPI ({public_url.public_url}) diputus dulu, supaya tidak berebut hostname dengan tunnel Streamlit.")
except Exception:
    print("(tidak ada tunnel FastAPI aktif untuk diputus, lanjut)")

def run_streamlit():
    os.system("streamlit run app.py")

threading.Thread(target=run_streamlit).start()
time.sleep(5)

public_url_streamlit = ngrok.connect(8501)
print(f"Streamlit is live at: {public_url_streamlit}")
print("(app.py memanggil FastAPI lewat http://localhost:8000, bukan lewat tunnel ini)")

In [ ]:
display_database_contents()

## Kesimpulan

- Jumlah mentah (raw count) bukan metrik yang bisa dibandingkan antar
  periode kalau volume chat-nya juga berubah — makanya rate (reaksi /
  total chat) ditambahkan.
- Cache yang "pintar" (fuzzy matching) berisiko salah kalau logikanya cuma
  ambil kecocokan pertama di atas threshold rendah, bukan yang terbaik —
  bug ini yang tadinya membuat cache menjawab pertanyaan yang salah.
- Feedback loop harus benar-benar menutup siklus: dislike/regenerate yang
  tidak memengaruhi apa yang disajikan lagi cuma jadi angka statistik,
  bukan mekanisme perbaikan.

---
*Notebook ini disusun sebagai bagian dari pembelajaran mandiri materi
"Online vs Offline Metrics for LLM Applications" — rubythalib.ai.*